# BT5151 Credit Risk Monitoring Pipeline

This notebook demonstrates the planned BT5151 multi-agent credit risk workflow on the credit score dataset.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn gradio langgraph pydantic openpyxl


In [ ]:
from pathlib import Path
import sys

import gradio as gr
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

sys.path.append(str(Path.cwd() / "src"))

from bt5151_credit_risk.business import explain_risk, recommend_action
from bt5151_credit_risk.config import GROUP_COLUMN, RANDOM_SEED, TARGET_COLUMN, TEST_SIZE
from bt5151_credit_risk.evaluate import choose_best_model, compute_multiclass_metrics
from bt5151_credit_risk.graph import build_graph
from bt5151_credit_risk.preprocess import preprocess_credit_data
from bt5151_credit_risk.profile import build_dataset_profile
from bt5151_credit_risk.train import build_candidate_models


In [ ]:
DATA_PATH = Path("train.csv")
df = pd.read_csv(DATA_PATH, low_memory=False)
dataset_profile = build_dataset_profile(df)
dataset_profile["row_count"], dataset_profile["target_distribution"]


In [ ]:
preprocess_result = preprocess_credit_data(df)
candidate_feature_frame = preprocess_result.feature_frame.select_dtypes(include=["number"]).fillna(0)
target_map = {"Poor": 0, "Standard": 1, "Good": 2}
y = preprocess_result.target.map(target_map)
groups = df[GROUP_COLUMN]

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(candidate_feature_frame, y, groups))

X_train = candidate_feature_frame.iloc[train_idx]
X_test = candidate_feature_frame.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

X_train.shape, X_test.shape


In [ ]:
models = build_candidate_models()
results = {}
fitted_models = {}
class_names = ["Poor", "Standard", "Good"]

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    metrics = compute_multiclass_metrics(y_test, predictions, class_names)
    results[model_name] = metrics
    fitted_models[model_name] = model

selection = choose_best_model(results)
selection


In [ ]:
best_model = fitted_models[selection["model_name"]]
sample_index = X_test.index[0]
sample_features = X_test.loc[[sample_index]]
sample_probabilities = best_model.predict_proba(sample_features)[0]
label_lookup = {0: "Poor", 1: "Standard", 2: "Good"}
predicted_code = int(best_model.predict(sample_features)[0])
predicted_label = label_lookup[predicted_code]
probability_map = {
    label_lookup[idx]: float(score)
    for idx, score in enumerate(sample_probabilities)
}
explanation = explain_risk(predicted_label, probability_map)
action = recommend_action(explanation)
explanation, action


In [ ]:
graph = build_graph()
sorted(graph.nodes.keys())


In [ ]:
def predict_from_row(row_index):
    row_index = int(row_index)
    row_features = candidate_feature_frame.loc[[row_index]].fillna(0)
    probabilities = best_model.predict_proba(row_features)[0]
    prediction = int(best_model.predict(row_features)[0])
    probability_map = {
        label_lookup[idx]: float(score)
        for idx, score in enumerate(probabilities)
    }
    explanation = explain_risk(label_lookup[prediction], probability_map)
    action = recommend_action(explanation)
    confidence = max(probability_map.values())
    return (
        explanation["summary"],
        action["action"],
        action["reason"],
        selection["model_name"],
        confidence,
    )


demo = gr.Interface(
    fn=predict_from_row,
    inputs=gr.Slider(minimum=0, maximum=len(candidate_feature_frame) - 1, step=1, value=0, label="Customer Row Index"),
    outputs=[
        gr.Textbox(label="Risk Summary"),
        gr.Textbox(label="Recommended Action"),
        gr.Textbox(label="Action Reason"),
        gr.Textbox(label="Selected Model"),
        gr.Number(label="Top Confidence"),
    ],
    title="BT5151 Credit Risk Monitor",
    description="Business-facing credit risk explanation and action recommendation.",
)

RUN_GRADIO = False
if RUN_GRADIO:
    demo.launch(share=True)
